In [43]:
from langchain.document_loaders import PyPDFLoader
## from langchain.document_loaders import TextLoader ## used for txt files
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, pipeline
from langchain import HuggingFacePipeline
from langchain.chains import RetrievalQA

In [ ]:
loader = PyPDFLoader("dataset/DeepSeek-V3.pdf") ##pdf loader
documents = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 150) ##splitting the text with 1000 characters in each split and 150 characters from previous chunk
docs = text_splitter.split_documents(documents)
texts = [doc.page_content for doc in docs]

In [ ]:
embedding_model = HuggingFaceBgeEmbeddings(model_name="all-MiniLM-L6-v2") ##applying the all-miniLM-L6-v2 embeddings on the dataset
vector_database = FAISS.from_documents(docs, embedding_model) ## making the vector database

In [ ]:
vector_database.save_local("database_deepseek") ## saving the database to use it in future

In [ ]:
vector_database = FAISS.load_local("database_deepseek", embedding_model, allow_dangerous_deserialization=True) ##loading the database
## allow_dangerous_deserialization is because it can be dangerous if there is some change in the file

In [ ]:
question = "conclusion of deepseek"
searchdocs = vector_database.similarity_search(question) ## Getting the chunks which matches our question
for doc in searchdocs:
    print(f"{doc.page_content}\n")

configuration is consistent with that used in DeepSeek-V2, being applied exclusively to the
decoupled shared key k𝑅
𝑡 . The hyper-parameters remain identical across both phases, with the
scale 𝑠= 40, 𝛼= 1, 𝛽 = 32, and the scaling factor √
𝑡 = 0.1 ln 𝑠+1. In the first phase, the sequence
length is set to 32K, and the batch size is 1920. During the second phase, the sequence length is
increased to 128K, and the batch size is reduced to 480. The learning rate for both phases is set
to 7.3 ×10−6, matching the final learning rate from the pre-training stage.
Through this two-phase extension training, DeepSeek-V3 is capable of handling inputs up to
128K in length while maintaining strong performance. Figure 8 illustrates that DeepSeek-V3,
following supervised fine-tuning, achieves notable performance on the "Needle In A Haystack"
(NIAH) test, demonstrating consistent robustness across context window lengths up to 128K.
23

4.4.2. Evaluation Results
In Table 3, we compare the base model of De

In [ ]:
model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding = True, truncation = True, max_length = 512)
question_answerer = pipeline(
    "question-answering",
    model = model_name,
    tokenizer=tokenizer,
    return_tensors = "pt"
)

llm = HuggingFacePipeline( ##hugging face pipeline makes it easier to integrate in the whole chain
    pipeline = question_answerer,
    model_kwargs= {"temperature": 0.7, "max_length": 512}
)

c:\Users\mayan\anaconda3\envs\ai\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mayan\.cache\huggingface\hub\models--Intel--dynamic_tinybert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP dow

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [41]:
retriever = vector_database.as_retriever(search_kwargs = {'k':10})
relevant_docs = retriever.get_relevant_documents("architecture of the deepseek model")
context = " ".join([doc.page_content for doc in relevant_docs])
context

'model architecture (Section 2). Subsequently, we introduce our infrastructures, encompassing\nour compute clusters, the training framework, the support for FP8 training, the inference\ndeployment strategy, and our suggestions on future hardware design. Next, we describe our\npre-training process, including the construction of training data, hyper-parameter settings, long-\ncontext extension techniques, the associated evaluations, as well as some discussions (Section 4).\nThereafter, we discuss our efforts on post-training, which include Supervised Fine-Tuning (SFT),\nReinforcement Learning (RL), the corresponding evaluations, and discussions (Section 5). Lastly,\nwe conclude this work, discuss existing limitations of DeepSeek-V3, and propose potential\ndirections for future research (Section 6).\n2. Architecture\nWe first introduce the basic architecture of DeepSeek-V3, featured by Multi-head Latent Atten- While acknowledging its strong performance and cost-effectiveness, we also reco

In [36]:
"""qa = RetrievalQA.from_chain_type(llm = llm, chain_type="refine", retriever = retriever, return_source_documents = False)
question = "what is the architecture of the deepseek model"
result = qa.run({"query" : question})
result"""

## not working here because of this llm

'qa = RetrievalQA.from_chain_type(llm = llm, chain_type="refine", retriever = retriever, return_source_documents = False)\nquestion = "what is the architecture of the deepseek model"\nresult = qa.run({"query" : question})\nresult'

In [42]:
result = question_answerer({
    "question": "explain the architecture of the deepseek model",
    "context": context 
})
result["answer"]

c:\Users\mayan\anaconda3\envs\ai\Lib\site-packages\transformers\pipelines\question_answering.py:390: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


'Figure 2 illustrates the basic architecture of DeepSeek-V3'